# Aula 3 — Lab de Agentes (CredSim)

Cliente HTTP que ataca a validação de documentos da CredSim — um **agente** que lê o conteúdo extraído (OCR) de um documento e decide uma ação. **Pré-requisito:** app no ar — na raiz do projeto:

```
docker compose up --build
```

Financeira A em http://localhost:8000. Estrutura: **cenário negativo** (vulnerável) → **cenário positivo** (mitigado), com evidência nos logs. Os documentos de exemplo estão em `lab/exemplos/`.

In [ ]:
import os, requests
BASE = os.environ.get('CREDSIM_URL', 'http://localhost:8000')

def set_defenses(input_validation=False, output_validation=False, least_privilege=False, api_security=False):
    return requests.post(BASE + '/api/defenses', json={
        'input_validation': input_validation, 'output_validation': output_validation,
        'least_privilege': least_privilege, 'api_security': api_security,
    }).json()

def validar_doc(content):
    return requests.post(BASE + '/api/validate-doc', json={'content': content}).json()

def mostrar_logs(n=6):
    for e in requests.get(BASE + '/api/logs').json()[:n]:
        flags = []
        if e.get('injection_detectada'): flags.append('INJEÇÃO DETECTADA')
        if e.get('auto_aprovado'): flags.append('AUTO-APROVADO')
        print('#' + str(e['seq']) + ' [' + e['stage'] + '] ' + (' '.join(flags) or '-'))

try:
    print('Conectado:', requests.get(BASE + '/api/info', timeout=3).json())
except Exception as e:
    print('App não respondeu — rode `docker compose up --build` na raiz.'); print(e)

## ❌ Cenário NEGATIVO — a injeção vira ação
O documento carrega uma instrução oculta; o agente validador a obedece: aprova sozinho e eleva o limite (**LLM01 indireta → LLM06 excessive agency**).

In [ ]:
with open('../exemplos/documento_envenenado.txt', encoding='utf-8') as f:
    doc_envenenado = f.read()

set_defenses()
r = validar_doc(doc_envenenado)
print(r['mensagem'])
print('Auto-aprovado?', r['auto_aprovado'], '| Ação executada:', r['acao_executada'])
print('\n--- evidência nos logs ---'); mostrar_logs(3)

### Controle — o documento legítimo
Sem instrução embutida, o mesmo agente só valida os dados — nenhuma ação automática.

In [ ]:
with open('../exemplos/documento_legitimo.txt', encoding='utf-8') as f:
    doc_legitimo = f.read()
r = validar_doc(doc_legitimo)
print(r['mensagem'], '| injeção detectada?', r['injection_detectada'])

## ✅ Cenário POSITIVO — separação de confiança
**Mitigação:** `input_validation` ON trata o conteúdo do documento como **DADO**, nunca como comando — a instrução é detectada, mas nenhuma ação é executada.

In [ ]:
set_defenses(input_validation=True)
r = validar_doc(doc_envenenado)
print(r['mensagem'])
print('Auto-aprovado?', r['auto_aprovado'], '| Ação executada:', r['acao_executada'])
print('\n--- evidência nos logs ---'); mostrar_logs(3)

## Conclusão
- **Negativo:** o agente não distingue dado de instrução — o documento manda, o agente obedece (excessive agency a partir de injeção indireta).
- **Positivo:** separar confiança (documento = dado) contém o ataque sem impedir a validação normal.
- Aprofundamento: o mesmo padrão (agente + ferramenta) aparece no pipeline de código (`05_pipeline_codigo.ipynb`) e no multi-agent (`04_multiagent.ipynb`); as defesas a fundo, na Aula 5.